# Download Historic Realtime Wind Data from NOAA

Description: https://www.ndbc.noaa.gov/station_realtime.php?station=burl1

In [1]:
import requests
import pandas as pd
from datetime import datetime
import re
from pathlib import Path
from zoneinfo import ZoneInfo


In [2]:
station = "burl1"
api = f"https://www.ndbc.noaa.gov/data/realtime2/{station.upper()}.cwind"
path = "./windData"

In [3]:
def lp(v):
    print(f"[{datetime.now()}] {v}")

lp("Starting...")

[2026-04-15 09:43:50.360915] Starting...


In [4]:
lp(api)

[2026-04-15 09:43:50.366682] https://www.ndbc.noaa.gov/data/realtime2/BURL1.cwind


In [5]:
resp = requests.get(api)

In [6]:
resp.raise_for_status()

In [7]:
data = resp.text.splitlines()
data[:5]

['#YY  MM DD hh mm WDIR WSPD GDR GST GTIME',
 '#yr  mo dy hr mn degT m/s degT m/s hhmm',
 '2026 04 15 14 00 141  7.2 140  7.7 1359',
 '2026 04 15 13 50 145  7.2 999 99.0 9999',
 '2026 04 15 13 40 147  6.2 999 99.0 9999']

In [8]:
reg = re.compile(r"\s+")
dataSplit = list(map(reg.split, data))
dataSplit[:5]

[['#YY', 'MM', 'DD', 'hh', 'mm', 'WDIR', 'WSPD', 'GDR', 'GST', 'GTIME'],
 ['#yr', 'mo', 'dy', 'hr', 'mn', 'degT', 'm/s', 'degT', 'm/s', 'hhmm'],
 ['2026', '04', '15', '14', '00', '141', '7.2', '140', '7.7', '1359'],
 ['2026', '04', '15', '13', '50', '145', '7.2', '999', '99.0', '9999'],
 ['2026', '04', '15', '13', '40', '147', '6.2', '999', '99.0', '9999']]

In [9]:
dataDf = pd.DataFrame(dataSplit[2:], columns=dataSplit[0]).astype({col:float for col in dataSplit[0]})

dataDf = dataDf.astype({col:int for col in dataDf.columns[ dataDf.mod(1).max() == 0 ].to_list()})
dataDf.head()    


,#YY,MM,DD,hh,mm,WDIR,WSPD,GDR,GST,GTIME
0,2026,4,15,14,0,141,7.2,140,7.7,1359
1,2026,4,15,13,50,145,7.2,999,99.0,9999
2,2026,4,15,13,40,147,6.2,999,99.0,9999
3,2026,4,15,13,30,151,6.2,999,99.0,9999
4,2026,4,15,13,20,149,6.7,999,99.0,9999


In [10]:
dataDf.dtypes

#YY        int64
MM         int64
DD         int64
hh         int64
mm         int64
WDIR       int64
WSPD     float64
GDR        int64
GST      float64
GTIME      int64
dtype: object

In [11]:
tz = ZoneInfo("America/Chicago")
utcTz = ZoneInfo("UTC")
def makeDate(row):
    
    dateOut = datetime(int(row['#YY']),\
                       int(row['MM']),\
                       int(row['DD']),\
                       int(row['hh']),\
                       int(row['mm']),\
                       tzinfo=utcTz).astimezone(tz)
    
    
    return dateOut
    
dataDf.index = dataDf.apply(makeDate, axis=1)

In [12]:
dataDf.head()

,#YY,MM,DD,hh,mm,WDIR,WSPD,GDR,GST,GTIME
2026-04-15 09:00:00-05:00,2026,4,15,14,0,141,7.2,140,7.7,1359
2026-04-15 08:50:00-05:00,2026,4,15,13,50,145,7.2,999,99.0,9999
2026-04-15 08:40:00-05:00,2026,4,15,13,40,147,6.2,999,99.0,9999
2026-04-15 08:30:00-05:00,2026,4,15,13,30,151,6.2,999,99.0,9999
2026-04-15 08:20:00-05:00,2026,4,15,13,20,149,6.7,999,99.0,9999


In [13]:
lp(f"Start: {dataDf.index.min()}\t\tStop: {dataDf.index.max()}")

[2026-04-15 09:43:50.824766] Start: 2026-02-28 18:00:00-06:00		Stop: 2026-04-15 09:00:00-05:00


In [14]:
Path(path).mkdir(exist_ok=True)

In [15]:
saveNameBase = f"{station}_{dataDf.index.max().strftime("%Y-%m-%d")}"
saveNamePickle = saveNameBase + ".pickle"
saveNameCSV = saveNameBase + ".csv"

lp(f"Saving to {saveNamePickle}")
dataDf.to_pickle(f"{path}/{saveNamePickle}")

lp(f"Saving to {saveNameCSV}")
dataDf.to_csv(f"{path}/{saveNameCSV}")


[2026-04-15 09:43:50.833959] Saving to burl1_2026-04-15.pickle
[2026-04-15 09:43:50.835114] Saving to burl1_2026-04-15.csv


In [16]:
lp("Finished")

[2026-04-15 09:43:50.906863] Finished
